In [ ]:
# input
file = "../../../_database/disprot/DisProt_release_2025_06_with_ambiguous_evidences.json"
pred_idp_file = "../../data/pred_ge_3_clique_3.tsv" # spatially adjacent
anno_file = "../../../collect_annotation/afdb_anno/data/entryId-repId-pfamId-tedId-annoLevel-sp.tsv"

# Critical assessment of protein intrinsic disorder prediction

In [ ]:
import pandas as pd
import json
with open(file, "r") as f:
    data = json.load(f)
data.keys()

df_pred = pd.read_table(pred_idp_file)
pred_ids = set(df_pred['seq_id'].map(lambda x: x.split("-")[1]))

df_anno = pd.read_table(anno_file, header=None)
no_anno_ids = set(df_anno[df_anno[4].isna()][0])
del df_anno

ids = set()
for i in data['data']:
    ids.add(i['acc'])

inter = pred_ids & ids & no_anno_ids
del ids

df_inter_pred = df_pred[df_pred['seq_id'].map(lambda x: x.split("-")[1] in inter)]

dict_keys(['data', 'size'])

In [ ]:
id2dis_info = dict()
for i in data['data']:
    if i['acc'] in inter:
        id2dis_info[i['acc']] = i

id2dis_area = dict()
for i in id2dis_info.keys():
    id = i
    result = [""] * 2700
    info = id2dis_info[id]['disprot_consensus']['full']
    for j in info:
        s = j['start']
        e = j['end']
        result[s - 1: e] = [j['type']] * (e - s + 1)
    id2dis_area[id] = result

In [42]:
def convert(row):
    seq_id = row['seq_id'].split("-")[1]
    area = id2dis_area[seq_id]
    posi = list(map(int, row['posi'].split(",")))
    return ",".join([area[p] for p in posi])

df_inter_pred['disprot_area'] = df_inter_pred.apply(lambda row: convert(row), axis=1)
len(df_inter_pred)

/tmp/ipykernel_60358/3756532780.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_inter_pred['disprot_area'] = df_inter_pred.apply(lambda row: convert(row), axis=1)


16

In [43]:
df_inter_pred[df_inter_pred['disprot_area'].map(lambda x: x.count("D") > 0 or x.count("T") > 0)]

,seq_id,avg_plddt,posi,pred,plddt,site,disprot_area
3041654,AFDB:AF-O60927-F1,66.322,"59,84,89","0.2577,0.8363,0.8436","67.64,54.12,63.88","59,84,89","D,D,D"
5157079,AFDB:AF-O86488-F1,72.325,"578,586,589,651,654,673,690,698,701,764,783,80...","0.9147,0.7419,0.9683,0.4818,0.7087,0.6991,0.93...","94.19,85.59,87.66,88.55,92.7,95.08,94.48,91.82...","578,586,589,654,673;872,875,894;801,805,809,81...","T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,..."
5477197,AFDB:AF-J7JU64-F1,52.872,"20,46,47,59","0.2194,0.608,0.5227,0.2272","72.64,41.74,40.22,37.48","46,47,59;20,46,47",",D,D,D"
6042257,AFDB:AF-P05367-F1,92.155,"54,80,92,96","0.9395,0.4965,0.3248,0.7041","98.57,92.67,92.05,96.55","54,80,92,96",",,D,D"
25951545,AFDB:AF-P0DJI8-F1,93.723,"54,80,92,96","0.9247,0.4392,0.2984,0.7023","98.45,92.35,92.08,96.24","54,80,92,96",",D,,"
26292554,AFDB:AF-Q9VVJ7-F1,81.011,"37,40,55,58","0.33,0.3028,0.5163,0.2546","87.8,89.7,90.65,90.41","37,40,55,58","D,D,D,D"


In [ ]:
# cands
# O60927 126
# J7JU64 68
# Q9VVJ7 178

{'full': [{'start': 89, 'end': 102, 'type': 'D'}],
 'Structural state': [{'start': 89, 'end': 102, 'type': 'D'}]}